In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tetapan lalai dan pilihan konfigurasi transpilasi


<details>
<summary><b>Package versions</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Litar abstrak perlu ditranspil kerana QPU mempunyai set gate asas yang terhad dan tidak boleh melaksanakan operasi sewenang-wenangnya. Fungsi transpiler adalah untuk mengubah litar sewenang-wenangnya supaya ia boleh dijalankan pada QPU yang ditentukan. Ini dilakukan dengan menterjemahkan litar ke gate asas yang disokong, dan dengan memasukkan gate SWAP yang diperlukan, supaya kesambungan litar sepadan dengan QPU.

Seperti yang dijelaskan dalam [Transpil dengan pengurus pass](transpile-with-pass-managers), anda boleh membuat [pengurus pass](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) menggunakan fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) dan menghantar litar atau senarai litar ke kaedah [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run)-nya untuk mentranspilnya. Anda boleh memanggil `generate_preset_pass_manager` dengan hanya menghantar tahap pengoptimuman dan backend, memilih untuk menggunakan lalai untuk semua pilihan lain, atau anda boleh menghantar argumen tambahan ke fungsi untuk menghaluskan transpilasi.

## Penggunaan asas tanpa parameter
Dalam contoh ini, kami menghantar litar dan QPU sasaran ke transpiler tanpa menentukan sebarang parameter tambahan.

Buat litar dan lihat hasilnya:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpil litar dan lihat hasilnya:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Semua parameter yang tersedia
Berikut adalah semua parameter yang tersedia untuk fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). Terdapat dua kelas argumen: yang menerangkan sasaran kompilasi, dan yang mempengaruhi cara transpiler berfungsi.

Semua parameter kecuali `optimization_level` adalah pilihan. Untuk butiran penuh, lihat [dokumentasi API Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - Berapa banyak pengoptimuman yang perlu dilakukan pada litar. Integer dalam julat (0 - 3). Tahap yang lebih tinggi menghasilkan litar yang lebih dioptimumkan, dengan kos masa transpilasi yang lebih lama. Lihat [Tetapkan tahap pengoptimuman transpiler](set-optimization) untuk butiran lanjut.

### Parameter yang digunakan untuk menerangkan sasaran kompilasi:
Argumen ini menerangkan QPU sasaran untuk pelaksanaan litar, termasuk maklumat seperti peta pasangan QPU (yang menerangkan kesambungan qubit), gate asas yang disokong oleh QPU, dan kadar ralat gate.

Banyak parameter ini diterangkan secara terperinci dalam [Parameter yang kerap digunakan untuk transpilasi](common-parameters).

<details>
  <summary>
    **Parameter QPU (`Backend`)**
  </summary>

**Parameter Backend** - Jika anda menentukan `backend`, anda tidak perlu menentukan `target` atau sebarang pilihan backend lain. Begitu juga, jika anda menentukan `target`, anda tidak perlu menentukan `backend` atau sebarang pilihan backend lain.
- `backend` (Backend) - Jika ini ditetapkan, transpiler mengkompilasi litar input ke peranti ini. Jika sebarang pilihan lain ditetapkan yang mempengaruhi tetapan ini, seperti `coupling_map`, ia mengatasi tetapan dari `backend`.
- `target` (Target) - Sasaran transpiler backend. Biasanya ini ditentukan sebagai sebahagian daripada argumen backend, tetapi jika anda membina objek Target secara manual, anda boleh menentukannya di sini. Ini mengatasi sasaran dari `backend`.
- `backend_properties` (BackendProperties) - Sifat yang dikembalikan oleh QPU, termasuk maklumat tentang ralat gate, ralat readout, masa koherensi qubit, dan sebagainya. Cari QPU yang menyediakan maklumat ini dengan menjalankan `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - Sekatan perkakasan kawalan pilihan pada resolusi masa arahan. Maklumat ini disediakan oleh konfigurasi QPU. Jika QPU tidak mempunyai sebarang sekatan pada peruntukan masa arahan, `timing_constraints` adalah `None` dan tiada pelarasan dilakukan. QPU mungkin melaporkan satu set sekatan, iaitu:
    - `granularity`: Nilai integer yang mewakili resolusi gate nadi minimum dalam unit dt. Gate nadi yang ditentukan oleh pengguna perlu mempunyai tempoh yang merupakan gandaan nilai granulariti ini.
    - `min_length`: Nilai integer yang mewakili panjang gate nadi minimum dalam unit dt. Gate nadi yang ditentukan oleh pengguna perlu lebih panjang daripada panjang ini.
    - `pulse_alignment`: Nilai integer yang mewakili resolusi masa masa mula arahan gate. Arahan gate perlu dimulakan pada masa yang merupakan gandaan nilai ini.
    - `acquire_alignment`: Nilai integer yang mewakili resolusi masa masa mula arahan ukur. Arahan ukur perlu dimulakan pada masa yang merupakan gandaan nilai ini.
</details>

<details>
  <summary>
    **Parameter susun atur dan topologi**
  </summary>

- `basis_gates` (List[str] | None) - Senarai nama gate asas untuk dibuka. Contohnya ['u1', 'u2', 'u3', 'cx']. Jika `None`, tidak dibuka.
- `coupling_map` (CouplingMap | List[List[int]]) - Peta pasangan terarah (mungkin tersuai) untuk disasarkan dalam pemetaan. Jika peta pasangan adalah simetri, kedua-dua arah perlu ditentukan. Format ini disokong:
    - Contoh CouplingMap
    - Senarai - mesti diberikan sebagai matriks kebersebelahan, di mana setiap entri menentukan semua interaksi dua qubit terarah yang disokong oleh QPU. Contohnya: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Pemetaan operasi litar kepada jadual nadi. Jika `None`, `instruction_schedule_map` QPU digunakan.
</details>

### Parameter yang digunakan untuk mempengaruhi cara transpiler berfungsi
Parameter ini mempengaruhi peringkat transpilasi tertentu. Sesetengahnya mungkin mempengaruhi berbilang peringkat, tetapi hanya disenaraikan di bawah satu peringkat untuk kesederhanaan. Jika anda menentukan argumen, seperti `initial_layout` untuk qubit yang anda ingin gunakan, nilai tersebut mengatasi semua pass yang boleh mengubahnya. Dengan kata lain, transpiler tidak akan mengubah apa-apa yang anda tentukan secara manual. Untuk butiran tentang peringkat tertentu, lihat [Peringkat transpiler](transpiler-stages).

<details>
  <summary>
    **Peringkat permulaan**
  </summary>

- `hls_config` (HLSConfig) - Kelas konfigurasi pilihan `HLSConfig` yang dihantar terus ke pass transformasi `HighLevelSynthesis`. Kelas konfigurasi ini membolehkan anda menentukan senarai algoritma sintesis dan parameternya untuk pelbagai objek tahap tinggi.
- `init_method` (str) - Nama plugin untuk digunakan bagi peringkat permulaan. Secara lalai, plugin luaran tidak digunakan. Anda boleh melihat senarai plugin yang dipasang dengan menjalankan `list_stage_plugins()` dengan `init` untuk argumen nama peringkat.
- `unitary_synthesis_method` (str) - Nama kaedah sintesis uniter untuk digunakan. Secara lalai, `default` digunakan. Anda boleh melihat senarai plugin yang dipasang dengan menjalankan `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - Kamus konfigurasi pilihan yang dihantar terus ke plugin sintesis uniter. Secara lalai tetapan ini tidak memberi kesan kerana kaedah sintesis uniter lalai tidak mengambil konfigurasi tersuai. Menggunakan konfigurasi tersuai hanya perlu apabila plugin sintesis uniter ditentukan dengan argumen `unitary_synthesis`. Oleh kerana ini adalah tersuai untuk setiap plugin sintesis uniter, rujuk dokumentasi plugin untuk cara menggunakan pilihan ini.
</details>

<details>
  <summary>
    **Peringkat susun atur**
  </summary>

- `initial_layout` (Layout | Dict | List) - Kedudukan awal qubit maya pada qubit fizikal. Jika susun atur ini menjadikan litar serasi dengan sekatan `coupling_map`, ia akan digunakan. Susun atur akhir tidak dijamin sama, kerana transpiler mungkin mengatur qubit melalui pertukaran atau cara lain. Untuk butiran penuh, lihat [bahagian Susun atur awal.](common-parameters#initial-layout)
- `layout_method` (str) - Nama pass pemilihan susun atur (`default`, `dense`, `sabre`, dan `trivial`). Ini juga boleh menjadi nama plugin luaran untuk digunakan bagi peringkat susun atur. Anda boleh melihat senarai plugin yang dipasang dengan menjalankan `list_stage_plugins()` dengan `layout` untuk argumen `stage_name`. Nilai lalainya ialah `sabre`.
</details>

<details>
  <summary>
    **Peringkat penghalaan**
  </summary>

- `routing_method` (str) - Nama pass penghalaan (`basic`, `lookahead`, `default`, `sabre`, atau `none`). Ini juga boleh menjadi nama plugin luaran untuk digunakan bagi peringkat penghalaan. Anda boleh melihat senarai plugin yang dipasang dengan menjalankan `list_stage_plugins()` dengan `routing` untuk argumen `stage_name`. Nilai lalainya ialah `sabre`.
</details>

<details>
  <summary>
    **Peringkat terjemahan**
  </summary>

- `translation_method` (str) - Nama pass terjemahan (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Ini juga boleh menjadi nama plugin luaran untuk digunakan bagi peringkat terjemahan. Anda boleh melihat senarai plugin yang dipasang dengan menjalankan `list_stage_plugins()` dengan `translation` untuk argumen `stage_name`. Nilai lalainya ialah `translator`.
</details>

<details>
  <summary>
    **Peringkat pengoptimuman**
  </summary>

- `approximation_degree` (float, dalam julat 0-1 | None) - Penentu heuristik yang digunakan untuk penghampiran litar (1.0 = tiada penghampiran, 0.0 = penghampiran maksimum). Nilai lalainya ialah 1.0. Menentukan `None` menetapkan darjat penghampiran kepada kadar ralat yang dilaporkan. Lihat [bahagian Darjat penghampiran](common-parameters#approx-degree) untuk butiran lanjut.
- `optimization_method` (str) - Nama plugin untuk digunakan bagi peringkat pengoptimuman. Secara lalai plugin luaran tidak digunakan. Anda boleh melihat senarai plugin yang dipasang dengan menjalankan `list_stage_plugins()` dengan `optimization` untuk argumen `stage_name`.
</details>

<details>
  <summary>
    **Peringkat penjadualan**
  </summary>

- `scheduling_method` (str) - Nama pass penjadualan. Ini juga boleh menjadi nama plugin luaran untuk digunakan bagi peringkat penjadualan. Anda boleh melihat senarai plugin yang dipasang dengan menjalankan `list_stage_plugins()` dengan `scheduling` untuk argumen `stage_name`.
  * 'as_soon_as_possible': Jadualkan arahan secara tamak, seawal mungkin pada sumber qubit (alias: `asap`).
  * 'as_late_as_possible': Jadualkan arahan lewat, iaitu, mengekalkan qubit dalam keadaan asas apabila boleh (alias: `alap`). Ini adalah lalai.
</details>

<details>
  <summary>
    **Pelaksanaan transpiler**
  </summary>

- `seed_transpiler` (int) - Menetapkan benih rawak untuk bahagian stokastik transpiler.
</details>

Nilai lalai berikut digunakan jika anda tidak menentukan sebarang parameter di atas. Rujuk [halaman rujukan API](../api/qiskit/transpiler_preset) kaedah ini untuk maklumat lanjut:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>